In [1]:
pip install pennylane 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 86.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
Adaptive Quantum Capsule Network (QCN) — sklearn Digits Binary Classification
==============================================================================
Architecture : N_CAPS capsules × CAP_SIZE qubits/capsule
Preprocessing:
    Raw image  : 8×8 = 64 pixels
    PCA        : 64  →  N_QUBITS components   (N_QUBITS = N_CAPS × CAP_SIZE)
    Reshape    : N_QUBITS  →  IMG_SIDE × IMG_SIDE  "compressed image"
    Scale      : MinMaxScaler  →  [0, π]  (angle encoding into qubits)

Outputs:
  training_log.csv
  effective_dimension.csv
  efficiency_analysis.csv
  capsule_correlation.csv
"""

import os
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# ═══════════════════════════════════════════════════════════════
# 0.  ALL HYPERPARAMETERS — change only here, everything derives
# ═══════════════════════════════════════════════════════════════

DIGIT_A     = 0              # first digit class
DIGIT_B     = 1              # second digit class

N_CAPS      = 4              # number of capsules
CAP_SIZE    = 4              # qubits per capsule
DEPTHS      = [4, 3, 3, 4]  # per-capsule circuit depth  (len must == N_CAPS)

N_EPOCHS    = 30             # training epochs
BATCH_SIZE  = 8              # mini-batch size
LR          = 0.05           # Adam learning rate

QUICK_EP    = 5              # epochs for baseline models in efficiency analysis
TEST_SIZE   = 0.25           # fraction of data held out for testing
RANDOM_SEED = 42

OUTPUT_DIR  = "/kaggle/working/"

# ── Derived constants (do NOT edit below this line) ─────────────
N_QUBITS    = N_CAPS * CAP_SIZE           # total qubits = PCA components
IMG_SIDE    = int(np.sqrt(N_QUBITS))      # compressed image side length
N_SAMPLES_ED = [100, 500, 1000, 5000, 10000]  # sample sizes for eff-dim

assert len(DEPTHS) == N_CAPS,       "len(DEPTHS) must equal N_CAPS"
assert IMG_SIDE ** 2 == N_QUBITS,   "N_QUBITS must be a perfect square"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print(f"  QCN Config")
print(f"  Digits       : {DIGIT_A} vs {DIGIT_B}")
print(f"  Capsules     : {N_CAPS}  x  {CAP_SIZE} qubits  =  {N_QUBITS} total")
print(f"  Depths       : {DEPTHS}")
print(f"  PCA shape    : 64  ->  {N_QUBITS}  ->  {IMG_SIDE}x{IMG_SIDE} image")
print(f"  Epochs       : {N_EPOCHS}   Batch: {BATCH_SIZE}   LR: {LR}")
print("=" * 60)

# ═══════════════════════════════════════════════════════════════
# 1.  DATASET  +  PCA PREPROCESSING
# ═══════════════════════════════════════════════════════════════
print(f"\n[Data] Loading sklearn digits ({DIGIT_A} vs {DIGIT_B}) ...")

digits = load_digits()
mask   = (digits.target == DIGIT_A) | (digits.target == DIGIT_B)
X_raw  = digits.data[mask]        # shape: (N, 64)  raw 8x8 pixels flattened
y_raw  = digits.target[mask]

# Binary labels:  DIGIT_A -> -1,   DIGIT_B -> +1
y = np.where(y_raw == DIGIT_A, -1.0, 1.0)

# ── PCA: 64 -> N_QUBITS components ──────────────────────────
pca     = PCA(n_components=N_QUBITS, random_state=RANDOM_SEED)
X_pca   = pca.fit_transform(X_raw)        # shape: (N, N_QUBITS)

# ── Scale each PCA component to [0, pi] ─────────────────────
scaler   = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)    # shape: (N, N_QUBITS)

# ── Reshape to IMG_SIDE x IMG_SIDE compressed image ─────────
# Each sample: flat N_QUBITS vector -> IMG_SIDE x IMG_SIDE grid
# The flat vector is what feeds into the circuit (one value per qubit)
X_images = X_scaled.reshape(-1, IMG_SIDE, IMG_SIDE)   # (N, 4, 4)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"  PCA variance explained : {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"  Train : {len(X_train)}   Test : {len(X_test)}")
print(f"  Input per sample       : flat {N_QUBITS} = {IMG_SIDE}x{IMG_SIDE} compressed image")

# ═══════════════════════════════════════════════════════════════
# 2.  QUANTUM DEVICE
# ═══════════════════════════════════════════════════════════════
dev = qml.device("default.qubit", wires=N_QUBITS)

# ═══════════════════════════════════════════════════════════════
# 3.  CIRCUIT BUILDER
# ═══════════════════════════════════════════════════════════════
def build_circuit(n_qubits, n_caps, cap_size, depths, device):
    """
    Build Adaptive QCN circuit.

    Param layout per capsule c:
        depths[c] x cap_size x 4   (3 Rot angles + 1 data-scaling angle)

    Returns
    -------
    circuit      : qml.QNode
    total_params : int
    offsets      : list[int]   param-slice boundaries per capsule
    """
    offsets = [0]
    for c in range(n_caps):
        offsets.append(offsets[-1] + depths[c] * cap_size * 4)
    total_params = offsets[-1]
    max_d        = max(depths)

    @qml.qnode(device, diff_method="best")
    def circuit(params, x):
        # ── Hadamard initialisation ──────────────────────────
        for i in range(n_qubits):
            qml.Hadamard(wires=i)

        for layer in range(max_d):
            # ── Intra-capsule layers ─────────────────────────
            for c in range(n_caps):
                if layer >= depths[c]:
                    continue

                w  = list(range(c * cap_size, (c + 1) * cap_size))
                cp = params[offsets[c]: offsets[c + 1]].reshape(
                         depths[c], cap_size, 4)

                # x is flat N_QUBITS vector; each capsule reads its slice
                xc = [float(x[c * cap_size + j]) for j in range(cap_size)]

                # Data re-uploading via RY
                for j, wire in enumerate(w):
                    qml.RY(xc[j] * cp[layer, j, 3], wires=wire)

                # Trainable Rot gates
                for j, wire in enumerate(w):
                    qml.Rot(cp[layer, j, 0],
                            cp[layer, j, 1],
                            cp[layer, j, 2], wires=wire)

                # Intra-capsule entanglement (even layers only)
                if layer % 2 == 0:
                    for j in range(len(w) - 1):
                        qml.CNOT(wires=[w[j], w[j + 1]])
                    if len(w) > 2:
                        qml.CNOT(wires=[w[-1], w[0]])

            # ── Inter-capsule boundary CNOTs (even layers only) ──
            if layer % 2 == 0:
                ic_offset = (layer // 2) % 2
                for c in range(ic_offset, n_caps - 1, 2):
                    if layer < depths[c] and layer < depths[c + 1]:
                        boundary_a = (c + 1) * cap_size - 1
                        boundary_b = (c + 1) * cap_size
                        qml.CNOT(wires=[boundary_a, boundary_b])

        # One expectation value per capsule (first qubit of each capsule)
        return qml.math.stack([
            qml.expval(qml.PauliZ(c * cap_size)) for c in range(n_caps)
        ])

    return circuit, total_params, offsets


circuit, total_params, offsets = build_circuit(
    N_QUBITS, N_CAPS, CAP_SIZE, DEPTHS, dev
)
print(f"\nCircuit ready  |  total_params = {total_params}")

# ═══════════════════════════════════════════════════════════════
# 4.  LOSS + PREDICTION HELPERS
# ═══════════════════════════════════════════════════════════════
def predict_one(params, x):
    """Scalar prediction in [-1, +1]: mean of capsule expectation values."""
    return qml.math.mean(circuit(params, x))


def mse_loss(params, X_batch, y_batch):
    preds = qml.math.stack([predict_one(params, x) for x in X_batch])
    return qml.math.mean((preds - y_batch) ** 2)


def accuracy(params, X, y):
    signs = np.array([float(np.sign(predict_one(params, x))) for x in X])
    signs = np.where(signs == 0, 1.0, signs)   # tie-break to +1
    return float(np.mean(signs == y))

# ═══════════════════════════════════════════════════════════════
# 5.  TRAINING LOOP
# ═══════════════════════════════════════════════════════════════
print(f"\n[Train] Starting training ({N_EPOCHS} epochs) ...")

np.random.seed(RANDOM_SEED)
params    = np.random.uniform(0, np.pi, total_params, requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=LR)

training_log = []

for epoch in range(1, N_EPOCHS + 1):
    perm   = np.random.permutation(len(X_train))
    X_shuf = X_train[perm]
    y_shuf = y_train[perm]

    epoch_loss, n_batches = 0.0, 0

    for start in range(0, len(X_train), BATCH_SIZE):
        Xb = X_shuf[start: start + BATCH_SIZE]
        yb = y_shuf[start: start + BATCH_SIZE]
        params, loss_val = optimizer.step_and_cost(
            lambda p: mse_loss(p, Xb, yb), params
        )
        epoch_loss += float(loss_val)
        n_batches  += 1

    avg_loss  = epoch_loss / n_batches
    train_acc = accuracy(params, X_train, y_train)
    test_acc  = accuracy(params, X_test,  y_test)

    training_log.append({
        "Epoch":     epoch,
        "Loss":      round(avg_loss,  6),
        "Train_Acc": round(train_acc, 4),
        "Test_Acc":  round(test_acc,  4),
    })
    print(f"  Epoch {epoch:>3}/{N_EPOCHS}  |  "
          f"loss={avg_loss:.4f}  "
          f"train_acc={train_acc:.3f}  "
          f"test_acc={test_acc:.3f}")

pd.DataFrame(training_log).to_csv(f"{OUTPUT_DIR}training_log.csv", index=False)
print(f"\n  Saved -> training_log.csv")

current_acc = accuracy(params, X_test, y_test)
print(f"\n  Final Test Accuracy : {current_acc*100:.2f}%")

# ═══════════════════════════════════════════════════════════════
# 6.  HELPER: GATE COUNT
# ═══════════════════════════════════════════════════════════════
def get_gate_counts(depths, n_caps, cap_size):
    total_ry   = sum(depths) * cap_size
    total_rot  = sum(depths) * cap_size
    total_cnot = sum(
        (cap_size if cap_size > 2 else cap_size - 1) * (d // 2 + 1)
        for d in depths
    )
    total_cnot += (n_caps - 1) * (max(depths) // 2)
    return total_ry + total_rot + total_cnot

# ═══════════════════════════════════════════════════════════════
# 7.  EXPERIMENT 1 — EFFECTIVE DIMENSION
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 1] Computing Effective Dimension ...")

F       = qml.gradients.quantum_fisher(circuit)(params, X_test[0])
trace_F = float(np.trace(F))
F_hat   = (total_params * F) / trace_F if trace_F > 0 else F

ed_results = []
for n in N_SAMPLES_ED:
    scale    = n / (2 * np.pi * np.log(n))
    mat      = np.eye(total_params) + scale * F_hat
    sign, ld = np.linalg.slogdet(mat)
    eff_dim  = float((2 * ld) / np.log(scale)) if sign > 0 else float("nan")
    ed_results.append({"Sample_Size": n, "Effective_Dimension": eff_dim})
    print(f"  n={n:>6}  eff_dim={eff_dim:.4f}")

pd.DataFrame(ed_results).to_csv(f"{OUTPUT_DIR}effective_dimension.csv", index=False)
print(f"  Saved -> effective_dimension.csv")

# ═══════════════════════════════════════════════════════════════
# 8.  EXPERIMENT 2 — EFFICIENCY ANALYSIS  (real trained accuracies)
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 2] Efficiency Analysis ...")

def train_quick(depths_cfg, n_ep=QUICK_EP):
    """Train a fixed-depth model for a few epochs and return test accuracy."""
    circ_tmp, tp, offsets_tmp = build_circuit(N_QUBITS, N_CAPS, CAP_SIZE, depths_cfg, dev)

    # Local predict/loss using circ_tmp, not the global circuit
    def predict_one_tmp(p, x):
        return qml.math.mean(circ_tmp(p, x))

    def mse_loss_tmp(p, X_batch, y_batch):
        preds = qml.math.stack([predict_one_tmp(p, x) for x in X_batch])
        return qml.math.mean((preds - y_batch) ** 2)

    def accuracy_tmp(p, X, y):
        signs = np.array([float(np.sign(predict_one_tmp(p, x))) for x in X])
        signs = np.where(signs == 0, 1.0, signs)
        return float(np.mean(signs == y))

    np.random.seed(RANDOM_SEED)
    p   = np.random.uniform(0, np.pi, tp, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=LR)

    for _ in range(n_ep):
        perm = np.random.permutation(len(X_train))
        for s in range(0, len(X_train), BATCH_SIZE):
            Xb = X_train[perm[s: s + BATCH_SIZE]]
            yb = y_train[perm[s: s + BATCH_SIZE]]
            # Capture Xb/yb properly with default args to avoid closure issues
            p, _ = opt.step_and_cost(
                lambda pp, _Xb=Xb, _yb=yb: mse_loss_tmp(pp, _Xb, _yb), p
            )

    return accuracy_tmp(p, X_test, y_test)   # use same circ_tmp, same p

depths_d2 = [2] * N_CAPS
depths_d4 = [4] * N_CAPS

print(f"  Training Fixed_D2 {depths_d2} ({QUICK_EP} epochs) ...")
acc_d2 = train_quick(depths_d2)
print(f"  Fixed_D2  test_acc={acc_d2:.3f}")

print(f"  Training Fixed_D4 {depths_d4} ({QUICK_EP} epochs) ...")
acc_d4 = train_quick(depths_d4)
print(f"  Fixed_D4  test_acc={acc_d4:.3f}")

efficiency_data = [
    {
        "Model":      f"Adaptive_QCapsule {DEPTHS}",
        "Gate_Count": get_gate_counts(DEPTHS,    N_CAPS, CAP_SIZE),
        "Accuracy":   round(current_acc, 4),
    },
    {
        "Model":      f"Fixed_QCapsule_D2 {depths_d2}",
        "Gate_Count": get_gate_counts(depths_d2, N_CAPS, CAP_SIZE),
        "Accuracy":   round(acc_d2, 4),
    },
    {
        "Model":      f"Fixed_QCapsule_D4 {depths_d4}",
        "Gate_Count": get_gate_counts(depths_d4, N_CAPS, CAP_SIZE),
        "Accuracy":   round(acc_d4, 4),
    },
]

pd.DataFrame(efficiency_data).to_csv(
    f"{OUTPUT_DIR}efficiency_analysis.csv", index=False
)
print(f"  Saved -> efficiency_analysis.csv")

# ═══════════════════════════════════════════════════════════════
# 9.  EXPERIMENT 3 — CAPSULE CORRELATION MATRIX
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 3] Capsule Correlation Matrix ...")

n_corr_samples  = min(50, len(X_test))
capsule_outputs = []
for x in X_test[:n_corr_samples]:
    out = circuit(params, x)
    capsule_outputs.append(np.array(out))

df_caps = pd.DataFrame(
    np.array(capsule_outputs),
    columns=[f"Capsule_{i}" for i in range(N_CAPS)]
)
corr_matrix = df_caps.corr()
corr_matrix.to_csv(f"{OUTPUT_DIR}capsule_correlation.csv")
print(f"  Saved -> capsule_correlation.csv")

# ═══════════════════════════════════════════════════════════════
# 10. DONE
# ═══════════════════════════════════════════════════════════════
print(f"""
All experiments complete.
  Digit pair        : {DIGIT_A} vs {DIGIT_B}
  PCA components    : {N_QUBITS}  ->  {IMG_SIDE}x{IMG_SIDE} compressed image
  Final Test Acc    : {current_acc*100:.2f}%
  Files saved in    : {OUTPUT_DIR}
    training_log.csv
    effective_dimension.csv
    efficiency_analysis.csv
    capsule_correlation.csv
""")

  QCN Config
  Digits       : 0 vs 1
  Capsules     : 4  x  4 qubits  =  16 total
  Depths       : [4, 3, 3, 4]
  PCA shape    : 64  ->  16  ->  4x4 image
  Epochs       : 30   Batch: 8   LR: 0.05

[Data] Loading sklearn digits (0 vs 1) ...
  PCA variance explained : 93.5%
  Train : 270   Test : 90
  Input per sample       : flat 16 = 4x4 compressed image

Circuit ready  |  total_params = 224

[Train] Starting training (30 epochs) ...
  Epoch   1/30  |  loss=0.8283  train_acc=0.930  test_acc=0.867
  Epoch   2/30  |  loss=0.6914  train_acc=0.933  test_acc=0.922
  Epoch   3/30  |  loss=0.6398  train_acc=0.952  test_acc=0.933
  Epoch   4/30  |  loss=0.5474  train_acc=0.970  test_acc=0.933
  Epoch   5/30  |  loss=0.5304  train_acc=0.948  test_acc=0.933
  Epoch   6/30  |  loss=0.5090  train_acc=0.952  test_acc=0.911
  Epoch   7/30  |  loss=0.4947  train_acc=0.952  test_acc=0.889
  Epoch   8/30  |  loss=0.4879  train_acc=0.967  test_acc=0.944
  Epoch   9/30  |  loss=0.4755  train_acc=0.956  